# **Finetune — train one CLIP-LoRA model on a split folder**

*   INPUT: an existing **split folder** (created by `Add_Split.ipynb`)
*   OUTPUT: one model folder inside `<SPLIT_DIR>/finetuned_models/<MODEL_NAME>/` containing:
    adapter weights, `config.json` (ALL hyperparameters + provenance + best-epoch results),
    `history.csv` (per-epoch logs), `training_curves.png`, `eval_test.csv`

**Hyper Parameters:**

*   SPLIT_DIR — which split to train on (models are stored INSIDE it)
*   DATASET — which dataset files to train with (`regular` / `all_nococo` / `all`)
*   LOSS + margin, anchor regularization (λ + per-text-type weights), LoRA, training config

**Sections:** 1 Utilities · 2 Model & training definitions · 3 Training (+ save) ·
4 Training results & curves · 5 Eval of THIS model on the test split (per category)

Data format (unified quads): one row = want `cos(source_a, close_a) > cos(source_b, far_b)`;
column names are ROLES (should-win pair / should-lose pair), not text types.


In [ ]:
# Hyper Parameters — everything tunable lives here

# --- Split folder (created by Add_Split.ipynb; REQUIRED to exist) ---
DRIVE_ROOT = '/content/drive/MyDrive/@@@@@@@@@@@@@@@@'
DATA_PATH  = 'ANON'
SPLIT_DIR  = f'{DRIVE_ROOT}/{DATA_PATH}/splits/Split_2'  # <-- cleaned benchmark

# --- Model identity ---
MODEL_NAME = None            # None = auto: '{DATASET}__{LOSS}__{regtag}'; or set custom
OVERWRITE  = False           # refuse to overwrite an existing model folder unless True

# --- Data mix (which dataset files to concatenate for training) ---
DATASET = 'regular_coco'     # 'regular' | 'regular_coco'  (zigzag/easy_* dropped, 2026-07)
DATASET_FILES = {
    'regular':      ['regular'],
    'regular_coco': ['regular', 'coco'],
    # dropped from the sweep (kept for reference):
    # 'all_nococo': ['regular', 'easy_close', 'easy_far', 'zigzag'],
    # 'all':        ['regular', 'easy_close', 'easy_far', 'zigzag', 'coco'],
}

# --- Loss ---
LOSS = 'triplet_fixed'       # 'triplet_fixed' | 'bpr'
TRIPLET_MARGIN = 0.2

# --- Anchor regularization (0 = off). Tightness of a text = LAMBDA * its type weight ---
LAMBDA_ANCHOR = 0.1          # grid winner (Split_2 sweep): lam=0.1, weights (1,1,1)
W_SOURCE, W_CLOSE, W_FAR = 1.0, 1.0, 1.0

# --- Coco amount (grid winner: aug 10% of the regular sample) ---
COCO_PCT = 0.10              # target coco rows as a fraction of the regular TRAIN rows;
                             # None = use coco_train.csv as-is. If the split's coco file
                             # has fewer rows than the target, uses all and warns.

# --- LoRA (text-tower attention q/k/v) ---
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.1

# --- Training ---
EPOCHS, LR, BATCH_SIZE = 10, 1e-4, 64
PATIENCE, WEIGHT_DECAY = 3, 0.01
SELECT_ON = 'regular'        # model selection on the HARD category's val pass_rate

# --- Final eval (Section 5) ---
EVAL_INCLUDE_BASELINE = True # add a CLIP-HF (no finetune) reference row — backbone is already loaded

SEED = 42
print("Hyperparameters set.")


Hyperparameters set.


# **Section 1: Utilities**

In [ ]:
# Section 1.1 — Pip installs (run once per Colab session)

!pip install -q transformers==4.44.0
# peft pinned: peft>=0.14 needs torch>=2.5 (torch.distributed.tensor) and crashes on Colab
!pip install -q "peft==0.13.2"
!pip uninstall -q -y torchao

print("Dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 146.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 137.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 15.6 MB/s eta 0:00:00
Dependencies installed.


In [ ]:
# Section 1.2 — Imports, Drive mount, load the split

import os, copy, json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

from google.colab import drive
drive.mount('/content/drive')
torch.manual_seed(SEED); np.random.seed(SEED)

# --- Load split meta + datasets ---
assert os.path.exists(f'{SPLIT_DIR}/split_meta.json'), f"not a split folder: {SPLIT_DIR}"
with open(f'{SPLIT_DIR}/split_meta.json') as fh:
    split_meta = json.load(fh)
print(f"Split: {split_meta['split_name']}  fingerprint={split_meta['fingerprint']}")

QUAD_COLS = ['source_a', 'close_a', 'source_b', 'far_b']

def load_split_files(split_lbl, cats):
    dfs = []
    for c in cats:
        p = f'{SPLIT_DIR}/{c}_{split_lbl}.csv'
        assert os.path.exists(p), f"missing dataset file: {p}"
        dfs.append(pd.read_csv(p, keep_default_na=False))
    return (pd.concat(dfs, ignore_index=True)
              .sample(frac=1, random_state=SEED).reset_index(drop=True))

cats = DATASET_FILES[DATASET]
df_train = load_split_files('train', cats)
df_val   = load_split_files('val',   cats)

# --- Retarget the coco fraction (the split folder was built at PCT_COCO=3%;
#     the grid winner used 10% of the regular count) ---
if COCO_PCT is not None and 'coco' in cats:
    def _retarget(df, lbl):
        coco, rest = df[df['category'] == 'coco'], df[df['category'] != 'coco']
        target = int(round(COCO_PCT * (df['category'] == 'regular').sum()))
        if len(coco) > target:
            coco = coco.sample(n=target, random_state=SEED)
        elif len(coco) < target:
            print(f"  [{lbl}] coco pool exhausted: target {target:,} rows "
                  f"(={COCO_PCT:.0%} of regular), file has {len(coco):,} -> using all")
        return (pd.concat([rest, coco], ignore_index=True)
                  .sample(frac=1, random_state=SEED).reset_index(drop=True))
    df_train = _retarget(df_train, 'train')
    df_val   = _retarget(df_val, 'val')

print(f"DATASET='{DATASET}' ({cats})  COCO_PCT={COCO_PCT}")
print(f"train: {len(df_train):,}  {df_train['category'].value_counts().to_dict()}")
print(f"val:   {len(df_val):,}  {df_val['category'].value_counts().to_dict()}")

# --- Resolve model name + output folder ---
def _regtag():
    if LAMBDA_ANCHOR == 0: return 'noreg'
    w = '-'.join(('%g' % x) for x in (W_SOURCE, W_CLOSE, W_FAR))
    return f'reg{LAMBDA_ANCHOR:g}_w{w}'

model_name = MODEL_NAME or f'{DATASET}__{LOSS}__{_regtag()}'
MODEL_DIR = f'{SPLIT_DIR}/finetuned_models/{model_name}'
if os.path.exists(f'{MODEL_DIR}/config.json') and not OVERWRITE:
    raise RuntimeError(f"Model already exists: {MODEL_DIR}\n"
                       "Set OVERWRITE=True or choose a different MODEL_NAME.")
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"\nModel: {model_name}\n  -> {MODEL_DIR}")


# **Section 2: Model & training definitions**

HF CLIP backbone + PEFT LoRA on text-tower attention (q/k/v, 12 layers), quad losses,
per-category evaluator, anchor-regularization helpers.

In [ ]:
# Section 2.1 — Backbone, LoRA, losses, evaluator, anchor helpers

from transformers import CLIPModel, CLIPTokenizer
from peft import (LoraConfig, get_peft_model, PeftModel,
                  get_peft_model_state_dict, set_peft_model_state_dict)

HF_CLIP_NAME   = "openai/clip-vit-base-patch32"
clip_tokenizer = CLIPTokenizer.from_pretrained(HF_CLIP_NAME)
clip_hf_model  = CLIPModel.from_pretrained(HF_CLIP_NAME).to(DEVICE).eval()   # frozen base (anchor target + baseline)
print("OK CLIP-HF backbone")

def clip_encode_text(model, texts, batch_size=256):
    """Any HF CLIPModel (base or PEFT) -> L2-normalized float32 numpy (N, 512)."""
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            chunk = list(texts[i:i + batch_size])
            inp = clip_tokenizer(chunk, padding=True, truncation=True,
                                 max_length=77, return_tensors="pt").to(DEVICE)
            feats = model.get_text_features(**inp)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            out.append(feats.float().cpu().numpy())
    return np.concatenate(out, axis=0)

def _embed_norm(model, texts):
    """Differentiable normalized embeddings (training path)."""
    inp = clip_tokenizer(list(texts), padding=True, truncation=True,
                         max_length=77, return_tensors="pt").to(DEVICE)
    feats = model.get_text_features(**inp)
    return feats / feats.norm(dim=-1, keepdim=True)

TEXT_ATTN_REGEX = r".*text_model\.encoder\.layers\.\d+\.self_attn\.(q_proj|k_proj|v_proj)"
def fresh_lora_clip():
    base = CLIPModel.from_pretrained(HF_CLIP_NAME).to(DEVICE)
    cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                     bias="none", target_modules=TEXT_ATTN_REGEX)
    return get_peft_model(base, cfg)

# --- Losses: want sim(source_a, close_a) > sim(source_b, far_b) ---
def bpr_loss(sim_pos, sim_neg):            return -F.logsigmoid(sim_pos - sim_neg).mean()
def triplet_loss(sim_pos, sim_neg):        return F.relu(TRIPLET_MARGIN - sim_pos + sim_neg).mean()
LOSSES = {'bpr': bpr_loss, 'triplet_fixed': triplet_loss}

# --- Dataset / collate (raw strings; tokenized per batch) ---
class QuadDataset(Dataset):
    def __init__(self, df):
        self.sa = df['source_a'].tolist(); self.ca = df['close_a'].tolist()
        self.sb = df['source_b'].tolist(); self.fb = df['far_b'].tolist()
    def __len__(self): return len(self.sa)
    def __getitem__(self, i): return self.sa[i], self.ca[i], self.sb[i], self.fb[i]

def quad_collate(batch):
    return ([b[0] for b in batch], [b[1] for b in batch],
            [b[2] for b in batch], [b[3] for b in batch])

# --- Per-category evaluator ---
def quad_pass_rates(encode_fn, df_q):
    uniq = list(dict.fromkeys(
        df_q['source_a'].tolist() + df_q['close_a'].tolist()
        + df_q['source_b'].tolist() + df_q['far_b'].tolist()))
    emb = encode_fn(uniq)
    m = {t: e for t, e in zip(uniq, emb)}
    A1 = np.stack([m[t] for t in df_q['source_a']]); B1 = np.stack([m[t] for t in df_q['close_a']])
    A2 = np.stack([m[t] for t in df_q['source_b']]); B2 = np.stack([m[t] for t in df_q['far_b']])
    d = df_q.copy(); d['correct'] = (A1 * B1).sum(-1) > (A2 * B2).sum(-1)
    out = {cat: float(sub['correct'].mean()) for cat, sub in d.groupby('category')}
    out['ALL'] = float(d['correct'].mean())
    return out

# --- Anchor regularization helpers ---
def precompute_base_embeddings(texts, batch_size=512):
    emb = clip_encode_text(clip_hf_model, texts, batch_size=batch_size)
    mat = torch.tensor(emb, device=DEVICE)
    return {t: mat[i] for i, t in enumerate(texts)}

def build_text_type_weights():
    """{text: weight} by text TYPE. Pools derived from regular+coco rows (their columns ARE
    text types); easy/zigzag rows reuse the same texts, so the pools cover everything."""
    both = pd.concat([df_train, df_val], ignore_index=True)
    typed = both[both['category'].isin(['regular', 'coco'])]
    src_pool   = set(both['source_a']) | set(both['source_b'])
    close_pool = set(typed['close_a'])
    far_pool   = set(typed['far_b'])
    w = {}
    for t in far_pool:   w[t] = W_FAR
    for t in close_pool: w[t] = W_CLOSE
    for t in src_pool:   w[t] = W_SOURCE   # highest priority
    return w

print("Definitions ready.")


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

OK CLIP-HF backbone
Definitions ready.


# **Section 3: Training**

Fresh pretrained CLIP + zero-init LoRA; within-batch text dedup; optional anchor term
(`loss = task + LAMBDA * weighted_drift_from_pretrained`); best snapshot by `SELECT_ON`
val pass_rate; everything saved into the model folder.

In [ ]:
# Section 3.1 — Train

model = fresh_lora_clip()
model.print_trainable_parameters()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=LR, weight_decay=WEIGHT_DECAY)
loader = DataLoader(QuadDataset(df_train), batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=quad_collate, num_workers=2, drop_last=True)
loss_fn = LOSSES[LOSS]

# drift thermometer (fixed val-text sample) + anchor-reg setup
_va_texts = list(dict.fromkeys(
    df_val['source_a'].tolist() + df_val['close_a'].tolist()
    + df_val['source_b'].tolist() + df_val['far_b'].tolist()))
drift_sample = _va_texts[:1500]
base_drift = clip_encode_text(clip_hf_model, drift_sample)

base_emb, type_w = None, {}
if LAMBDA_ANCHOR > 0:
    _tr_texts = list(dict.fromkeys(
        df_train['source_a'].tolist() + df_train['close_a'].tolist()
        + df_train['source_b'].tolist() + df_train['far_b'].tolist()))
    base_emb = precompute_base_embeddings(_tr_texts)
    type_w = build_text_type_weights()
    print(f"Anchor reg: lambda={LAMBDA_ANCHOR}, weights src/close/far = "
          f"{W_SOURCE}/{W_CLOSE}/{W_FAR}, {len(_tr_texts):,} base embeddings precomputed")

best, best_ep, best_state, no_imp, history = -1.0, -1, None, 0, []
t_start = datetime.now()

for epoch in range(1, EPOCHS + 1):
    model.train(); run_task, run_anchor, nb = 0.0, 0.0, 0
    for sa, ca, sb, fb in loader:
        texts = list(dict.fromkeys(sa + ca + sb + fb))          # within-batch dedup
        emb = _embed_norm(model, texts)
        idx = {t: i for i, t in enumerate(texts)}
        e1 = emb[[idx[t] for t in sa]]; f1 = emb[[idx[t] for t in ca]]
        e2 = emb[[idx[t] for t in sb]]; f2 = emb[[idx[t] for t in fb]]
        task = loss_fn((e1 * f1).sum(-1), (e2 * f2).sum(-1))
        loss = task
        if LAMBDA_ANCHOR > 0:
            base_mat = torch.stack([base_emb[t] for t in texts])
            w = torch.tensor([type_w.get(t, 1.0) for t in texts], device=DEVICE)
            drift = 1.0 - (emb * base_mat).sum(-1)
            anchor = (w * drift).sum() / w.sum()
            loss = task + LAMBDA_ANCHOR * anchor
            run_anchor += anchor.item()
        opt.zero_grad(); loss.backward(); opt.step()
        run_task += task.item(); nb += 1

    pr = quad_pass_rates(lambda t: clip_encode_text(model, t), df_val)
    cur = clip_encode_text(model, drift_sample)
    drift_val = float((1.0 - (cur * base_drift).sum(-1)).mean())
    sel = pr.get(SELECT_ON, pr['ALL'])
    history.append({'epoch': epoch, 'train_loss': run_task / max(nb, 1),
                    'anchor_loss': run_anchor / max(nb, 1), 'val_drift': drift_val, **pr})
    improved = sel > best
    if improved:
        best, best_ep = sel, epoch
        best_state = copy.deepcopy(get_peft_model_state_dict(model)); no_imp = 0
    else:
        no_imp += 1
    cats_str = ' '.join(f'{k}={v:.3f}' for k, v in pr.items() if k != 'ALL')
    print(f"ep{epoch:2d} loss={run_task/max(nb,1):.4f} anchor={run_anchor/max(nb,1):.4f} "
          f"drift={drift_val:.4f} | {cats_str}" + ("  *best*" if improved else ""))
    if no_imp >= PATIENCE:
        print(f"early stop at ep{epoch}")
        break

# --- Save: weights + history + config ---
set_peft_model_state_dict(model, best_state)
model.save_pretrained(MODEL_DIR)
hist_df = pd.DataFrame(history)
hist_df.to_csv(f'{MODEL_DIR}/history.csv', index=False)

best_row = hist_df[hist_df['epoch'] == best_ep].iloc[0].to_dict()
config = {
    'model_name': model_name,
    'split': {'name': split_meta['split_name'], 'fingerprint': split_meta['fingerprint'],
              'dir': SPLIT_DIR},
    'dataset': DATASET, 'dataset_files': cats,
    'n_train': len(df_train), 'n_val': len(df_val),
    'loss': LOSS, 'triplet_margin': TRIPLET_MARGIN,
    'anchor': {'lambda': LAMBDA_ANCHOR, 'w_source': W_SOURCE,
               'w_close': W_CLOSE, 'w_far': W_FAR},
    'lora': {'rank': LORA_RANK, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
             'backbone': HF_CLIP_NAME},
    'training': {'epochs_max': EPOCHS, 'epochs_ran': len(hist_df), 'lr': LR,
                 'batch_size': BATCH_SIZE, 'patience': PATIENCE,
                 'weight_decay': WEIGHT_DECAY, 'select_on': SELECT_ON, 'seed': SEED},
    'best': {'epoch': best_ep, **{k: v for k, v in best_row.items() if k != 'epoch'}},
    'runtime_minutes': round((datetime.now() - t_start).total_seconds() / 60, 1),
    'finished_at': datetime.now().isoformat(),
}
with open(f'{MODEL_DIR}/config.json', 'w') as fh:
    json.dump(config, fh, indent=2)

print(f"\nSaved to {MODEL_DIR}:")
print(f"  adapter weights | history.csv ({len(hist_df)} epochs) | config.json")
print(f"  best epoch {best_ep}: {SELECT_ON}={best:.4f}")


# **Section 4: Training results & curves**

In [ ]:
# Section 4.1 — Curves (train/anchor loss, drift, per-category val pass_rate)

cat_cols = [c for c in hist_df.columns
            if c not in ('epoch', 'train_loss', 'anchor_loss', 'val_drift')]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
ax.plot(hist_df['epoch'], hist_df['train_loss'], marker='o', label='task loss')
if hist_df['anchor_loss'].abs().sum() > 0:
    ax.plot(hist_df['epoch'], hist_df['anchor_loss'], marker='s', label='anchor loss')
ax.set_title(f'{model_name} — losses'); ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(hist_df['epoch'], hist_df['val_drift'], marker='o', color='purple')
ax.set_title('val drift (1 - cos vs pretrained)'); ax.set_xlabel('epoch'); ax.grid(alpha=0.3)

ax = axes[2]
for c in cat_cols:
    ax.plot(hist_df['epoch'], hist_df[c], marker='.', label=c)
ax.axvline(best_ep, color='gray', ls=':', alpha=0.6, label=f'best ep{best_ep}')
ax.set_title('val pass_rate per category'); ax.set_xlabel('epoch')
ax.set_ylim(0.4, 1.02); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved: {MODEL_DIR}/training_curves.png")


# **Section 5: Evaluate THIS model on the test split**

Only the just-trained model (plus an optional CLIP-HF reference row) — the full multi-encoder
comparison lives in the Evaluations notebook. Uses ALL `*_test.csv` files of the split,
regardless of the training DATASET, so you always see every test column.

In [ ]:
# Section 5.1 — Per-category test metrics + per-metadata breakdown for the finetuned model

ALL_CATS = ['regular', 'easy_close', 'easy_far', 'zigzag', 'coco']
df_test = pd.concat([pd.read_csv(f'{SPLIT_DIR}/{c}_test.csv', keep_default_na=False)
                     for c in ALL_CATS if os.path.exists(f'{SPLIT_DIR}/{c}_test.csv')],
                    ignore_index=True)
print(f"Test: {len(df_test):,} quads  {df_test['category'].value_counts().to_dict()}\n")

def per_row_eval(encode_fn, df_q):
    """Embed once; return df_q + sim_pos/sim_neg/margin/correct per row."""
    uniq = list(dict.fromkeys(
        df_q['source_a'].tolist() + df_q['close_a'].tolist()
        + df_q['source_b'].tolist() + df_q['far_b'].tolist()))
    emb = np.asarray(encode_fn(uniq), dtype=np.float32)
    m = {t: e for t, e in zip(uniq, emb)}
    d = df_q.copy()
    d['sim_pos'] = (np.stack([m[t] for t in d['source_a']])
                    * np.stack([m[t] for t in d['close_a']])).sum(-1)
    d['sim_neg'] = (np.stack([m[t] for t in d['source_b']])
                    * np.stack([m[t] for t in d['far_b']])).sum(-1)
    d['margin'] = d['sim_pos'] - d['sim_neg']; d['correct'] = d['margin'] > 0
    return d

def summarize(d, label):
    rows = []
    for cat, sub in list(d.groupby('category')) + [('ALL', d)]:
        n = len(sub); n_ok = int(sub['correct'].sum()); n_bad = n - n_ok
        ok, bad = sub['correct'].values, ~sub['correct'].values
        rows.append({'model': label, 'category': cat, 'n': n,
            'pass_rate': n_ok / n if n else float('nan'),
            'avg_correct_margin':  sub.loc[ok,  'margin'].mean()   if n_ok  else float('nan'),
            'avg_wrong_margin':   (-sub.loc[bad, 'margin']).mean() if n_bad else float('nan'),
            'avg_sim_pos_all': sub['sim_pos'].mean(), 'avg_sim_neg_all': sub['sim_neg'].mean()})
    return pd.DataFrame(rows)

# finetuned model (best snapshot already restored in Section 3) + optional baseline
d_ft = per_row_eval(lambda t: clip_encode_text(model, t), df_test)
row_models = {model_name: d_ft}
parts = [summarize(d_ft, model_name)]
if EVAL_INCLUDE_BASELINE:
    d_base = per_row_eval(lambda t: clip_encode_text(clip_hf_model, t), df_test)
    row_models['CLIP-HF'] = d_base
    parts.append(summarize(d_base, 'CLIP-HF'))
eval_df = pd.concat(parts, ignore_index=True).set_index(['model', 'category'])
eval_df.to_csv(f'{MODEL_DIR}/eval_test.csv')

# display: pass_rate pivot + full table
pr_pivot = eval_df['pass_rate'].unstack('category')
order = [c for c in ALL_CATS + ['ALL'] if c in pr_pivot.columns]
try:
    from IPython.display import display
    print("Test pass_rate per category:")
    display(pr_pivot[order].style.format('{:.3f}')
            .background_gradient(cmap='RdYlGn', vmin=0.4, vmax=1.0, axis=None))
    print("\nFull metrics:")
    display(eval_df.style.format({'pass_rate': '{:.1%}', 'avg_correct_margin': '{:.4f}',
                                  'avg_wrong_margin': '{:.4f}', 'avg_sim_pos_all': '{:.4f}',
                                  'avg_sim_neg_all': '{:.4f}'}))
except Exception:
    print(pr_pivot[order].to_string()); print(eval_df.to_string())

# --- Per-metadata breakdown on the REGULAR test rows (they carry full metadata) ---
# Answers "which enrichment types / rewrite rules / anchors are hard to rank?" per model.
META_DIMS = ['enrichment_label_close', 'enrichment_label_far',
             'rule_type_close', 'rule_type_far', 'anchor_id_close', 'anchor_id_far']
meta_rows = []
for dim in META_DIMS:
    if dim not in d_ft.columns:
        continue
    cols = {}
    for label, d in row_models.items():
        reg = d[(d['category'] == 'regular') & (d[dim].astype(str).str.len() > 0)]
        if not reg.empty:
            cols[label] = reg.groupby(dim)['correct'].mean()
    if not cols:
        continue
    reg_ft = d_ft[(d_ft['category'] == 'regular') & (d_ft[dim].astype(str).str.len() > 0)]
    tbl = pd.DataFrame(cols)
    tbl.insert(0, 'n', reg_ft.groupby(dim).size())
    tbl = tbl.sort_values(model_name)
    print(f"\n[regular] pass_rate by {dim}:")
    print(tbl.to_string(float_format=lambda x: f'{x:.3f}'))
    for val, r in tbl.iterrows():
        meta_rows.append({'dimension': dim, 'value': val, **r.to_dict()})
if meta_rows:
    pd.DataFrame(meta_rows).to_csv(f'{MODEL_DIR}/eval_test_by_metadata.csv', index=False)
    print(f"\nSaved: {MODEL_DIR}/eval_test_by_metadata.csv")

print(f"\nSaved: {MODEL_DIR}/eval_test.csv")
